In [25]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.home() / "documents/personal/Awslearning/smartstore-ai"

DATA_FILE = (
    PROJECT_ROOT
    / "data/raw/packsense/india_products.csv"
)
print(DATA_FILE)


df = pd.read_csv(DATA_FILE, dtype={"code": str})

print("Records:", len(df))
print("Columns:", len(df.columns))

/Users/vphari/documents/personal/Awslearning/smartstore-ai/data/raw/packsense/india_products.csv
Records: 25209
Columns: 36


In [26]:
print("Total records:", len(df))
print("Unique product codes:", df["code"].nunique())

print("Null product codes:", df["code"].isna().sum())

print(
    "Duplicate product codes:",
    df["code"].duplicated().sum()
)

print(
    "Blank product codes:",
    df["code"].astype(str).str.strip().eq("").sum()
)

Total records: 25209
Unique product codes: 25209
Null product codes: 0
Duplicate product codes: 0
Blank product codes: 0


In [27]:
def valid_values(series):
    values = series.astype("string").str.strip()

    return (
        series.notna()
        & values.ne("")
        & values.str.lower().ne("unknown")
    )

index_fields = [
    "product_name",
    "pnns_groups_1",
    "pnns_groups_2",
    "brands"
]

for field in index_fields:
    valid = valid_values(df[field])

    print(
        f"{field:20} "
        f"{valid.sum():6} valid records "
        f"({valid.mean()*100:.1f}%)"
    )

product_name          21999 valid records (87.3%)
pnns_groups_1          8159 valid records (32.4%)
pnns_groups_2          8159 valid records (32.4%)
brands                17155 valid records (68.1%)


In [28]:
duplicates = df[
    df["code"].duplicated(keep=False)
].sort_values("code")

display(
    duplicates[
        ["code", "product_name", "brands"]
    ].head(30)
)

,code,product_name,brands


In [29]:
eligibility = pd.DataFrame({
    "product_name": valid_values(df["product_name"]),
    "category": valid_values(df["pnns_groups_1"]),
    "subcategory": valid_values(df["pnns_groups_2"]),
    "brand": valid_values(df["brands"])
})

eligibility["category_gsi"] = (
    eligibility["product_name"]
    & eligibility["category"]
)

eligibility["subcategory_gsi"] = (
    eligibility["product_name"]
    & eligibility["subcategory"]
)

eligibility["brand_gsi"] = (
    eligibility["product_name"]
    & eligibility["brand"]
)

display(eligibility.sum().to_frame("eligible_records"))

,eligible_records
product_name,21999
category,8159
subcategory,8159
brand,17155
category_gsi,7684
subcategory_gsi,7684
brand_gsi,16841


In [30]:
print("Total records:", len(df))
print("Unique codes:", df["code"].nunique())
print("Duplicate codes:", df["code"].duplicated().sum())
print("Null codes:", df["code"].isna().sum())

print("\nMost common category values:")
display(df["pnns_groups_1"].value_counts(dropna=False).head(20))

print("\nMost common subcategory values:")
display(df["pnns_groups_2"].value_counts(dropna=False).head(20))

Total records: 25209
Unique codes: 25209
Duplicate codes: 0
Null codes: 0

Most common category values:


pnns_groups_1
unknown                    16434
Sugary snacks               2041
Cereals and potatoes        1982
Beverages                   1183
Milk and dairy products      851
Fat and sauces               741
Salty snacks                 738
NaN                          616
Fruits and vegetables        412
Composite foods               78
Fish Meat Eggs                74
Alcoholic beverages           40
Baby foods                    19
Name: count, dtype: int64


Most common subcategory values:


pnns_groups_2
unknown                             16434
Biscuits and cakes                   1196
Cereals                               709
NaN                                   616
Sweets                                593
Sweetened beverages                   507
Milk and yogurt                       468
Legumes                               409
Dressings and sauces                  398
Bread                                 396
Fats                                  343
Appetizers                            328
Breakfast cereals                     316
Nuts                                  275
Chocolate products                    239
Teas and herbal teas and coffees      210
Ice cream                             209
Fruits                                176
Artificially sweetened beverages      158
Potatoes                              152
Name: count, dtype: int64

In [31]:
import boto3

dynamodb = boto3.client(
    "dynamodb",
    endpoint_url="http://localhost:8001",
    region_name="ap-south-1",
    aws_access_key_id="local",
    aws_secret_access_key="local"
)

print("DynamoDB Local connection successful")

print(
    "Existing tables:",
    dynamodb.list_tables()["TableNames"]
)

DynamoDB Local connection successful
Existing tables: []


In [32]:
TABLE_NAME = "smartstore-dev-products"

existing_tables = dynamodb.list_tables()["TableNames"]

if TABLE_NAME in existing_tables:
    print(f"{TABLE_NAME} already exists.")
    print(dynamodb.describe_table(
        TableName=TABLE_NAME
    )["Table"]["KeySchema"])
else:
    print(f"{TABLE_NAME} is ready to be created.")

smartstore-dev-products is ready to be created.


In [34]:

import pandas as pd

path = "data/raw/packsense/india_products.csv"
df = pd.read_csv(path, dtype={"code": str}, nrows=5)

print("COLUMN NAMES:")
for column in df.columns:
    print(f"{column}: {df[column].dtype}")

print("\nSAMPLE RECORD:")
print(df.iloc[0].to_dict())


COLUMN NAMES:
code: object
product_name: object
brands: object
quantity_amount: float64
quantity_unit: object
product_quantity: float64
product_quantity_unit: object
serving_amount: float64
serving_unit: float64
pnns_groups_1: object
pnns_groups_2: object
nutrition_format: object
nutrition_per: float64
nutrition_preparation: object
energy_kj: float64
energy_kcal: float64
fat: float64
saturated_fat: float64
carbohydrates: float64
sugars: float64
fiber: float64
proteins: float64
salt: float64
sodium: float64
nutriscore_grade: object
nova_group: float64
environmental_score_grade: object
categories_tags: object
labels_tags: float64
allergens_tags: float64
traces_tags: float64
open_error_flags: int64
open_warning_flags: int64
open_info_flags: int64
existence_status: object
last_modified_t: int64

SAMPLE RECORD:
{'code': '0000000000000', 'product_name': nan, 'brands': nan, 'quantity_amount': nan, 'quantity_unit': nan, 'product_quantity': nan, 'product_quantity_unit': nan, 'serving_amount': n